<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Lane: Refresh / Content Opportunity Scoring

Research question: Can a leakage-safe machine-learning model rank content pages for search-performance review more effectively than a transparent rule-based baseline, using only signals available at the decision moment?

Decision supported: Help a human reviewer decide which eligible content pages to inspect first for a refresh, search-result presentation review, or monitoring.

Unit of analysis: One pseudonymized content page for one decision period.
Success criterion: Compare the baseline and ML rankers on the same held-out clients using Precision@K. Do not claim causal refresh impact or that the model explains Google's ranking algorithm.

In [1]:
# Research question and decision
PROJECT_TITLE = "Leakage-Safe Content Refresh Opportunity Prioritization"
LANE = "Refresh / Content Opportunity Scoring"
RESEARCH_QUESTION = (
    "Can a leakage-safe model rank content pages for search-performance review "
    "more effectively than a transparent rule-based baseline?"
)
UNIT_OF_ANALYSIS = "One pseudonymized content page for one decision period."
DECISION = "Which pages should a human reviewer inspect first for refresh, CTR review, or monitoring?"

print("Title:", PROJECT_TITLE)
print("Lane:", LANE)
print("Research question:", RESEARCH_QUESTION)
print("Unit:", UNIT_OF_ANALYSIS)
print("Decision supported:", DECISION)

Title: Leakage-Safe Content Refresh Opportunity Prioritization
Lane: Refresh / Content Opportunity Scoring
Research question: Can a leakage-safe model rank content pages for search-performance review more effectively than a transparent rule-based baseline?
Unit: One pseudonymized content page for one decision period.
Decision supported: Which pages should a human reviewer inspect first for refresh, CTR review, or monitoring?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Source: FlyRank/internship-warehouse on Hugging Face (gated release).

Tables to inspect: fact_content_daily_performance, fact_content_query_90d, dim_content, and dim_clients where relevant. Use only fields needed for the stated question.

Date windows: The notebook starts with February 2026 as an example feature window and March 2026 as an example outcome window. Confirm that both months exist and that coverage is sufficient before using them; change the windows if the warehouse schema/coverage requires it.

Availability rule: A missing row or unavailable measurement is not automatically zero performance. Use the warehouse's availability indicators and document any exclusion rules.

Public-safety rule: Do not publish client names, domains, page URLs, private search queries, credentials, raw exports, or row-level data that could identify a client.

In [3]:
# Data source and initial inspection
# This notebook queries the gated FlyRank warehouse with DuckDB.
# Do not print tokens or publish raw rows, private queries, client identifiers, or URLs.

%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib


from huggingface_hub import whoami, auth_check
from google.colab import userdata

# Load the token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

# Step 1: Verify your Hugging Face identity
user_info = whoami(token=HF_TOKEN)
print("Hugging Face identity:", user_info.get("name", "Authenticated"))

# Step 2: Check access to the DATASET repository
auth_check(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Successfully authenticated and verified dataset access!")

Hugging Face identity: anangshachatterjee
Successfully authenticated and verified dataset access!


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

1. Define the eligible page universe using only information available at the decision moment.
2. Aggregate feature-window search/content signals at the page level.
3. Construct the outcome from a later window, after checking coverage and missingness semantics.
4. Exclude target-derived flags and any future-window fields from features.
5. Compare a transparent baseline with Logistic Regression and Random Forest.
6. Hold out entire clients for testing; use the same held-out rows for every ranker.
7. Report Precision@K as the primary ranking metric, with PR-AUC/ROC-AUC as optional secondary metrics where valid.
8. Fix random seeds and record the exact windows, features, split, and exclusions.

The actual warehouse schema must be inspected before implementing the final aggregation and label. Do not fill in unverified column names or report results until the relevant cells have been run.


In [4]:
# Methodology plan and schema-aware guardrails
# IMPORTANT: Do not model until the schema and coverage output above have been inspected.
# This cell records the intended design and defines reusable leakage checks.

FEATURE_MONTH = "2026-02"   # Example only: verify this month exists and has adequate coverage.
OUTCOME_MONTH = "2026-03"   # Example only: verify this month exists and has adequate coverage.
TEST_CLIENT_FRACTION = 0.20

FORBIDDEN_FEATURE_NAMES = {
    "trend_direction", "trend_pct", "is_declining", "is_declining_label",
    "health_score", "is_quick_win", "needs_ctr_fix", "needs_engagement_fix",
    "is_underperformer", "is_initial_refresh_candidate", "recommended_action",
    "action_type", "label", "target", "outcome"
}

def leakage_audit(feature_columns, forbidden=FORBIDDEN_FEATURE_NAMES):
    """Raise an error if a forbidden/target-derived column appears in the feature list."""
    bad = sorted(
        c for c in feature_columns
        if c.lower() in forbidden
        or any(token in c.lower() for token in ["future_", "next_month", "outcome_"])
    )
    if bad:
        raise ValueError(f"Potential leakage features detected: {bad}")
    print("Leakage name check passed for", len(feature_columns), "candidate features.")

print("Feature window:", FEATURE_MONTH)
print("Outcome window:", OUTCOME_MONTH)
print("Validation design: hold out entire clients; never randomly split page rows.")
print("Planned models: transparent baseline, Logistic Regression, Random Forest.")
print("Primary metric: Precision@K on the same held-out clients.")
print(
    "Label must be constructed from the future window only after confirming data "
    "availability semantics. Missing/unavailable measurements must not be treated as zero."
)

Feature window: 2026-02
Outcome window: 2026-03
Validation design: hold out entire clients; never randomly split page rows.
Planned models: transparent baseline, Logistic Regression, Random Forest.
Primary metric: Precision@K on the same held-out clients.
Label must be constructed from the future window only after confirming data availability semantics. Missing/unavailable measurements must not be treated as zero.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Report only measured results from the executed notebook. Use the same held-out clients and the same positive-label definition for all methods.

The primary comparison is Precision@10, Precision@20, Precision@50, and Precision@100 where the test-set size permits. Include a table and a chart. Add PR-AUC and ROC-AUC only if the label and test set support them.

Do not insert illustrative numbers as results. If a model fails, record the failure and fix the pipeline before reporting.

In [5]:
# Results: comparison framework
# Run this after you have constructed a leakage-safe modelling table and evaluated each
# ranker on the SAME held-out clients. This cell intentionally does not invent metrics.

def precision_at_k(y_true, scores, k):
    """Precision among the k highest-scored rows. Ties are handled by stable row order."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    if len(y) != len(s):
        raise ValueError("y_true and scores must have the same length.")
    if len(y) == 0:
        return np.nan
    k = min(int(k), len(y))
    order = np.argsort(-s, kind="mergesort")[:k]
    return float(y[order].mean())

def compare_rankers(y_test, ranked_scores, ks=(10, 20, 50, 100)):
    """
    ranked_scores: dict of {ranker_name: score_array}; all arrays must use the same test rows.
    """
    rows = []
    for name, scores in ranked_scores.items():
        row = {"ranker": name}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(y_test, scores, k)
        rows.append(row)
    return pd.DataFrame(rows)

print("Evaluation helpers are ready.")
print("After training, call compare_rankers(y_test, {'Baseline': ..., 'Logistic Regression': ..., 'Random Forest': ...}).")

Evaluation helpers are ready.
After training, call compare_rankers(y_test, {'Baseline': ..., 'Logistic Regression': ..., 'Random Forest': ...}).


## 5. Limitations

*What this work cannot claim.*

This is an observational, retrospective prioritization experiment. It can measure how well a score ranks a defined future outcome in the available data, but it cannot establish that refreshing content causes performance to improve.

Coverage, seasonality, client mix, search demand, SERP changes, and measurement quality may affect the results. The target is an operational proxy, not ground truth for the ideal editorial action. A high score means “review sooner,” not “change this page automatically.” The analysis does not prove or reproduce Google's ranking algorithm.

In [6]:
# Limitations and honest framing
limitations = [
    "The data are observational; associations do not establish that refreshing a page causes better performance.",
    "A future-window label is a proxy for the review decision, not a direct measure of the causal effect of a content refresh.",
    "Search demand, seasonality, SERP changes, tracking coverage, and client-specific context may affect observed metrics.",
    "Coverage can vary by client and date. Unavailable measurements must be distinguished from true zero activity.",
    "Performance on held-out clients may differ from performance on clients represented during training.",
    "The model is decision support: a human reviewer should inspect recommendations before taking action.",
    "This analysis does not reveal or prove Google's ranking algorithm."
]
for i, item in enumerate(limitations, 1):
    print(f"{i}. {item}")

1. The data are observational; associations do not establish that refreshing a page causes better performance.
2. A future-window label is a proxy for the review decision, not a direct measure of the causal effect of a content refresh.
3. Search demand, seasonality, SERP changes, tracking coverage, and client-specific context may affect observed metrics.
4. Coverage can vary by client and date. Unavailable measurements must be distinguished from true zero activity.
5. Performance on held-out clients may differ from performance on clients represented during training.
6. The model is decision support: a human reviewer should inspect recommendations before taking action.
7. This analysis does not reveal or prove Google's ranking algorithm.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The final queue should contain only anonymized content identifiers and decision-support fields such as:
- rank and score/probability;
- reason code based on verified pre-decision signals;
- suggested review action;
- a short confidence/guardrail note.

Use action categories such as Review refresh opportunity, Review CTR/SERP presentation, and Monitor / gather more evidence only when supported by available fields. Human review is required. Do not include client IDs, names, domains, URLs, or private queries in the public paper.

In [7]:
# Ranked recommendations: action playbook
# Use anonymized content IDs only in private analysis. Never publish client names, domains,
# page URLs, private queries, credentials, or raw exports.

ACTION_PLAYBOOK = {
    "HIGH_SCORE": {
        "action": "Review refresh opportunity",
        "reason": "High model score; inspect content accuracy, freshness, and search intent fit.",
        "guardrail": "Do not refresh automatically; confirm with a human reviewer."
    },
    "HIGH_VISIBILITY_LOW_CTR": {
        "action": "Review title/meta description and SERP presentation",
        "reason": "Visible page with comparatively weak click capture, if supported by the verified data.",
        "guardrail": "Account for position, query mix, device, and seasonality before changing metadata."
    },
    "STALE_CONTENT": {
        "action": "Review factual freshness",
        "reason": "Content has not been updated recently, if a valid update-age field exists.",
        "guardrail": "Age alone does not prove that content is outdated."
    },
    "LOW_CONFIDENCE": {
        "action": "Monitor / gather more evidence",
        "reason": "Signals are weak, sparse, or coverage is uncertain.",
        "guardrail": "Do not treat missing measurements as poor performance."
    }
}

for code, item in ACTION_PLAYBOOK.items():
    print(f"\\n{code}: {item['action']}")
    print("Reason:", item["reason"])
    print("Guardrail:", item["guardrail"])

\nHIGH_SCORE: Review refresh opportunity
Reason: High model score; inspect content accuracy, freshness, and search intent fit.
Guardrail: Do not refresh automatically; confirm with a human reviewer.
\nHIGH_VISIBILITY_LOW_CTR: Review title/meta description and SERP presentation
Reason: Visible page with comparatively weak click capture, if supported by the verified data.
Guardrail: Account for position, query mix, device, and seasonality before changing metadata.
\nSTALE_CONTENT: Review factual freshness
Reason: Content has not been updated recently, if a valid update-age field exists.
Guardrail: Age alone does not prove that content is outdated.
\nLOW_CONFIDENCE: Monitor / gather more evidence
Reason: Signals are weak, sparse, or coverage is uncertain.
Guardrail: Do not treat missing measurements as poor performance.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper should contain aggregate, public-safe artifacts:
1. A model-vs-baseline Precision@K chart.
2. An aggregate comparison table with test-set metrics.
3. A feature importance or coefficient chart, after leakage review.
4. A small anonymized ranked-action example or summary by reason code.
5. A reproducibility section with repository/notebook links and the exact data windows.

Save charts under work/figures/ and keep raw datasets and row-level warehouse exports out of Git.

In [8]:
# Artifacts for the deployed research paper
# The following plotting helpers work once `comparison_df` contains evaluated model results.
# Save only aggregate charts/tables suitable for public release.

OUTPUT_DIR = "work/outputs"
FIGURE_DIR = "work/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

def plot_precision_at_k(comparison_df, output_path=None):
    metric_cols = [c for c in comparison_df.columns if c.startswith("Precision@")]
    if not metric_cols:
        raise ValueError("No Precision@K columns found in comparison_df.")
    long_df = comparison_df.melt(
        id_vars="ranker", value_vars=metric_cols,
        var_name="metric", value_name="precision"
    )
    long_df["k"] = long_df["metric"].str.replace("Precision@", "", regex=False).astype(int)
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, group in long_df.groupby("ranker"):
        group = group.sort_values("k")
        ax.plot(group["k"], group["precision"], marker="o", label=name)
    ax.set_title("Precision at K by ranking method")
    ax.set_xlabel("K (number of pages reviewed)")
    ax.set_ylabel("Precision@K")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()

print("Output folders created:", OUTPUT_DIR, "and", FIGURE_DIR)
print("When comparison_df exists, save the plot with:")
print("plot_precision_at_k(comparison_df, 'work/figures/precision_at_k.png')")
print("Also include an aggregate model-comparison table and an anonymized top-K action queue.")

Output folders created: work/outputs and work/figures
When comparison_df exists, save the plot with:
plot_precision_at_k(comparison_df, 'work/figures/precision_at_k.png')
Also include an aggregate model-comparison table and an anonymized top-K action queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.